## 1. Kiểm tra GPU

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "GPU not available")

## 2. Clone repo từ GitHub

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/social-media-mining.git"  # thay bằng URL repo thực
REPO_DIR = "/kaggle/working/social-media-mining"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

print("Repo ready at:", REPO_DIR)

## 3. Cài đặt dependencies

In [ ]:
os.system(
    f"pip install -q -r {REPO_DIR}/round-1-annotation/requirements.txt"
)
print("Dependencies installed.")

## 4. Cấu hình đường dẫn và HF_TOKEN

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Lấy HF_TOKEN từ Kaggle Secrets (Add-ons > Secrets)
try:
    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    print("HF_TOKEN not found in Secrets — model download may fail for gated models.")

# Trỏ HuggingFace cache về working dir để tránh hết dung lượng
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])

## 5. Kết nối thư mục data

Data phải được upload lên Kaggle Dataset và attach vào notebook này.
Dataset phải chứa các file/thư mục: `data-sample.json`, `data.json`, `images/`, `images-sample/`.
Sau khi attach, dataset sẽ được mount tại `/kaggle/input/TEN_DATASET/`.

In [ ]:
import shutil

# Tên dataset đã attach (thay bằng tên thực)
KAGGLE_DATASET_NAME = "visomd-data"  # ví dụ: nếu dataset mount tại /kaggle/input/visomd-data/
DATASET_PATH = f"/kaggle/input/{KAGGLE_DATASET_NAME}"

DATA_LINK = os.path.join(REPO_DIR, "data")

if os.path.islink(DATA_LINK):
    os.unlink(DATA_LINK)
elif os.path.isdir(DATA_LINK):
    shutil.rmtree(DATA_LINK)

os.symlink(DATASET_PATH, DATA_LINK)
print(f"Symlink: {DATA_LINK} -> {DATASET_PATH}")

# Kiểm tra
import glob
files = glob.glob(DATA_LINK + "/*")
print("Files in data/:", [os.path.basename(f) for f in files[:10]])

## 6. Chạy pipeline trên data mẫu (data-sample.json)

In [ ]:
PIPELINE_DIR = os.path.join(REPO_DIR, "round-1-annotation")
INPUT_DATA   = os.path.join(REPO_DIR, "data", "data-sample.json")
CONFIG       = os.path.join(PIPELINE_DIR, "configs", "round1.yaml")
OUTPUT_DIR   = "/kaggle/working/outputs_sample"

cmd = (
    f"cd {PIPELINE_DIR} && "
    f"python -m src.pipeline_round1 "
    f"--input_data {INPUT_DATA} "
    f"--config {CONFIG} "
    f"--output_dir {OUTPUT_DIR}"
)
print("Running:", cmd)
os.system(cmd)

## 7. Chạy pipeline trên toàn bộ data (data.json)

In [ ]:
INPUT_DATA_FULL = os.path.join(REPO_DIR, "data", "data.json")
OUTPUT_DIR_FULL = "/kaggle/working/outputs_full"

cmd_full = (
    f"cd {PIPELINE_DIR} && "
    f"python -m src.pipeline_round1 "
    f"--input_data {INPUT_DATA_FULL} "
    f"--config {CONFIG} "
    f"--output_dir {OUTPUT_DIR_FULL}"
)
print("Running:", cmd_full)
os.system(cmd_full)

## 8. Xem kết quả

In [ ]:
import json

stats_path = os.path.join(OUTPUT_DIR_FULL, "round1_stats.json")
if os.path.exists(stats_path):
    with open(stats_path, encoding="utf-8") as f:
        stats = json.load(f)
    print(json.dumps(stats, ensure_ascii=False, indent=2))
else:
    print("Stats file not found — pipeline may not have completed.")

In [ ]:
# Xem 5 record đầu tiên của round1_auto_accepted.jsonl
auto_path = os.path.join(OUTPUT_DIR_FULL, "round1_auto_accepted.jsonl")
if os.path.exists(auto_path):
    with open(auto_path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))
else:
    print("Output file not found.")

## 9. Lưu kết quả ra file để download

Sau khi notebook chạy xong, vào tab **Output** bên phải để download các file từ `/kaggle/working/outputs_full/`.

In [ ]:
import shutil

# Nén toàn bộ output thành một file zip để dễ download
zip_path = "/kaggle/working/round1_outputs"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR_FULL)
print("Saved:", zip_path + ".zip")